# Relax Structure with CHGNet — Crystal Hamiltonian Graph Network

This notebook demonstrates structural relaxation using **CHGNet**,
a graph neural network interatomic potential that models the universal potential
energy surface of the Materials Project (MP) dataset.

CHGNet predicts energies, forces, stresses, and magnetic moments,
making it well-suited for ionic and magnetic systems.


## 1. Set Input Parameters
### 1.1. Structure and Relaxation


In [ ]:
FOLDER = "uploads"
STRUCTURE_NAME = "Interface"  # Name of the structure to load from local file

RELAXATION_PARAMETERS = {
    "FMAX": 0.05,
}


## 2. Install Packages


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples|torch|chgnet")


In [ ]:
from mat3ra.notebooks_utils.pyodide.packages.torch import apply_all_patches

apply_all_patches(include_chgnet=True)


## 3. Load Materials


In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.standata.materials import Materials

structure = load_material_from_folder(FOLDER, STRUCTURE_NAME) or Material.create(
    Materials.get_by_name_first_match(STRUCTURE_NAME))

print(f"Structure: {structure.name}")
print(f"Formula: {structure.formula}")


### 3.1. Visualize Input Structure


In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

visualize(structure, repetitions=[1, 1, 1], rotation="-90x")


## 4. Apply Relaxation
### 4.1. Load CHGNet Model and Create Calculator


In [ ]:
from chgnet.model.model import CHGNet
from chgnet.model.dynamics import CHGNetCalculator

# Load the pretrained CHGNet v0.3.0 model
chgnet = CHGNet.load(model_name="0.3.0", use_device="cpu")
calculator = CHGNetCalculator(model=chgnet, use_device="cpu")

print(f"CHGNet v0.3.0 loaded ({sum(p.numel() for p in chgnet.parameters()):,} parameters)")


### 4.2. Relax with CHGNet


In [ ]:
import plotly.graph_objs as go
from IPython.display import display
from plotly.subplots import make_subplots

from mat3ra.made.tools.convert import to_ase
from ase.optimize import BFGS

ase_structure = to_ase(structure)
ase_structure.calc = calculator
dyn = BFGS(ase_structure)

steps = []
energies = []
forces_max = []

# Store original structure
ase_original_structure = ase_structure.copy()

def log_step():
    e = ase_structure.get_potential_energy()
    f = ase_structure.get_forces()
    fmax = (f**2).sum(axis=1).max()**0.5
    steps.append(len(steps))
    energies.append(e)
    forces_max.append(fmax)
    print(f"Step {len(steps)-1:3d}: E={e:.4f} eV  Fmax={fmax:.4f} eV/Å")

dyn.attach(log_step, interval=1)
log_step()  # log initial state
dyn.run(fmax=RELAXATION_PARAMETERS["FMAX"], steps=200)

ase_final_structure = ase_structure.copy()

# Plot convergence
fig = make_subplots(rows=1, cols=2, subplot_titles=("Energy", "Max Force"))
fig.add_trace(go.Scatter(x=steps, y=energies, mode="lines+markers", name="Energy (eV)"), row=1, col=1)
fig.add_trace(go.Scatter(x=steps, y=forces_max, mode="lines+markers", name="Fmax (eV/Å)"), row=1, col=2)
fig.update_layout(height=350, showlegend=False)
fig.show()


## 5. Analyze Results
### 5.1. View Structure Before and After Relaxation


In [ ]:
from mat3ra.made.tools.convert import from_ase

material_original = Material.create(from_ase(ase_original_structure))
material_relaxed = Material.create(from_ase(ase_final_structure))
material_original.name = structure.name
material_relaxed.name = structure.name + " (CHGNet Relaxed)"

visualize([material_original, material_relaxed], repetitions=[1, 1, 1], rotation="-90x")


### 5.2. Output interlayer distance before and after relaxation


In [ ]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance

SUBSTRATE_TAG = 0
FILM_TAG = 1

print(
    f"Interlayer distance before relaxation: {get_average_interlayer_distance(material_original, SUBSTRATE_TAG, FILM_TAG):.4f} Å")
print(
    f"Interlayer distance after relaxation:  {get_average_interlayer_distance(material_relaxed, SUBSTRATE_TAG, FILM_TAG):.4f} Å")


## References

[1] CHGNet: https://github.com/CederGroupHub/chgnet

[2] Bowen Deng et al., "CHGNet as a pretrained universal neural network potential for charge-informed atomistic modelling," Nature Machine Intelligence (2023)
